In [1]:
from pyspark.sql import SparkSession
import glob



In [2]:
sc=SparkSession.builder.getOrCreate()
input="C:/dewengineering/Mycodes/BroadcastLogs_2018_Q3_M8_sample.csv"
input_file = glob.glob(input)
df=sc.read.option("delimiter", "|").option("header", True).csv(input, inferSchema=True)
df.show(truncate=False)
# df.printSchema ()
df.dtypes

+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------------------------------------+-------------------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate   |SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration           |EndTime            |LogEntryDate|ProductionNO|ProgramTitle                                      |StartTime          |Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Lan

[('BroadcastLogID', 'int'),
 ('LogServiceID', 'int'),
 ('LogDate', 'date'),
 ('SequenceNO', 'int'),
 ('AudienceTargetAgeID', 'int'),
 ('AudienceTargetEthnicID', 'int'),
 ('CategoryID', 'int'),
 ('ClosedCaptionID', 'int'),
 ('CountryOfOriginID', 'int'),
 ('DubDramaCreditID', 'int'),
 ('EthnicProgramID', 'int'),
 ('ProductionSourceID', 'int'),
 ('ProgramClassID', 'int'),
 ('FilmClassificationID', 'int'),
 ('ExhibitionID', 'int'),
 ('Duration', 'timestamp'),
 ('EndTime', 'timestamp'),
 ('LogEntryDate', 'date'),
 ('ProductionNO', 'string'),
 ('ProgramTitle', 'string'),
 ('StartTime', 'timestamp'),
 ('Subtitle', 'string'),
 ('NetworkAffiliationID', 'int'),
 ('SpecialAttentionID', 'int'),
 ('BroadcastOriginPointID', 'int'),
 ('CompositionID', 'int'),
 ('Producer1', 'string'),
 ('Producer2', 'string'),
 ('Language1', 'int'),
 ('Language2', 'int')]

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df.select([count (when (col(c).isNull(), c)).alias (c) for c in df.columns]).show()

# df.withColumn(count (when (col(c).isNull(), c)).alias (c) for c in df.columns).show()


+--------------+------------+-------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+--------+-------+------------+------------+------------+---------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration|EndTime|LogEntryDate|ProductionNO|ProgramTitle|StartTime|Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|
+--------------+------------+-------+----------+-------------------+----------------------+----------+---------------+-----------------+----

In [4]:
from pyspark.sql.functions import isnan, when, count, col
from pyspark.sql.functions import mean, median, mode
#define function to fill null values with column mean

def fillna_mean(df, include=set()):
    means=df.agg(*(
    mean(x).alias (x) for x in df.columns if x in include ))

    return df.fillna(means.first().asDict())

def fillna_meadian(df, include=set()):
    medians=df.agg(*(
    median(x).alias (x) for x in df.columns if x in include ))

    return df.fillna(medians.first().asDict())


def fillna_mode(df, include=set()):
    modes=df.agg(*(
    mode(x).alias (x) for x in df.columns if x in include ))

    return df.fillna(modes.first().asDict())




In [5]:
#fill null values with mean,median,mode in specific columns
df = fillna_mean (df, ['AudienceTargetAgeID', 'AudienceTargetEthnicID', 'Language1', 'Language2','ClosedCaptionID','ProgramTitle','BroadcastOriginPointID'])
df = fillna_meadian(df, ['CategoryID', 'CountryOfOriginID','ExhibitionID', 'Subtitle', 'NetworkAffiliationID', 'SpecialAttentionID', 'CompositionID'])
df = fillna_mode (df, ['DubDramaCreditID','EthnicProgramID','ProductionSourceID', 'FilmClassificationID', 'ProductionNO', 'Broadcast OriginPointID', 'ClosedCaptionID", "ProgramTitle'])

df.show (10,truncate=False)

+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------------------------------------+-------------------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate   |SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration           |EndTime            |LogEntryDate|ProductionNO|ProgramTitle                                      |StartTime          |Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Lan

In [6]:
from pyspark.sql import Window


# windowspecl=Window.partitionBy("BroadcastLogID").orderBy(df["BroadcastLogID"]).rowsBetween(Window.unboundedPreceding,Window.currentRow)
# windowspec2=Window.partitionBy("BroadcastLogID").orderBy(df["BroadcastLogID"]).rowsBetween(Window.currentRow, Window.unboundedFollowing)




windowspecl=Window.partitionBy("SequenceNO").orderBy("BroadcastLogID").rowsBetween(Window.unboundedPreceding,Window.currentRow)
windowspec2=Window.partitionBy("SequenceNO").orderBy("BroadcastLogID").rowsBetween(Window.currentRow, Window.unboundedFollowing)



# windowspecl=Window.partitionBy("LogDate").orderBy("BroadcastLogID").rowsBetween(Window.unboundedPreceding,Window.currentRow)
# windowspec2=Window.partitionBy("LogDate").orderBy("BroadcastLogID").rowsBetween(Window.currentRow, Window.unboundedFollowing)


# windowspecl=Window.orderBy("BroadcastLogID").rowsBetween(Window.unboundedPreceding,Window.currentRow)
# windowspec2=Window.orderBy("BroadcastLogID").rowsBetween(Window.currentRow, Window.unboundedFollowing)



df=df.withColumn ("Producer1", last ("Producer1", ignorenulls=True).over (windowspecl))\
        .withColumn ("Producer2", last("Producer2", ignorenulls=True).over(windowspecl))\
        .withColumn ("Duration", last("Duration", ignorenulls=True).over(windowspecl))\
        .withColumn ("EndTime", last("EndTime", ignorenulls=True).over(windowspecl))


df=df.withColumn ("Producer1", last ("Producer1", ignorenulls=True).over (windowspec2))\
        .withColumn ("Producer2", last("Producer2", ignorenulls=True).over(windowspec2))\
        .withColumn ("Duration", last("Duration", ignorenulls=True).over(windowspec2))\
        .withColumn ("EndTime", last("EndTime", ignorenulls=True).over(windowspec2))


# # define the window
# window = Window.partitionBy('location')\
#                .orderBy('time')\
#                .rowsBetween(-sys.maxsize, 0)

# # define the forward-filled column
# filled_column = last(spark_df['temperature'], ignorenulls=True).over(window)

# # do the fill
# spark_df_filled = spark_df.withColumn('temp_filled_spark', filled_column)






df.show(10)
df.count()


+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------+-------------------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|   LogDate|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|           Duration|            EndTime|LogEntryDate|ProductionNO|        ProgramTitle|          StartTime|Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|
+--------------+------------+----------+----------+-

238945

In [7]:
df.select([count (when (col(c).isNull(), c)).alias (c) for c in df.columns]).show()


+--------------+------------+-------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+--------+-------+------------+------------+------------+---------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration|EndTime|LogEntryDate|ProductionNO|ProgramTitle|StartTime|Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|
+--------------+------------+-------+----------+-------------------+----------------------+----------+---------------+-----------------+----

In [8]:
df = fillna_mode (df, ['Producer1','Producer2'])
df.select([count (when (col(c).isNull(), c)).alias (c) for c in df.columns]).show()


+--------------+------------+-------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+--------+-------+------------+------------+------------+---------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration|EndTime|LogEntryDate|ProductionNO|ProgramTitle|StartTime|Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|
+--------------+------------+-------+----------+-------------------+----------------------+----------+---------------+-----------------+----

In [9]:
# duplicates = df.groupBy(df.columns)\
#     .agg(count("*").alias("count"))\
#     .filter(col("count") > 1)
# # show the results
# duplicates.show()


df_duplicates = df.groupBy(df.SequenceNO).count().filter("count > 1")
df_duplicates.show()
df_duplicates.count()

+----------+-----+
|SequenceNO|count|
+----------+-----+
|       148|  281|
|       463|  238|
|       471|  238|
|       496|  233|
|       833|  158|
|      1088|   71|
|      1238|   17|
|       243|  275|
|       392|  254|
|       540|  229|
|       623|  208|
|       737|  191|
|       858|  153|
|       897|  143|
|      1025|   87|
|      1084|   71|
|      1127|   53|
|        31|  317|
|       516|  232|
|      1139|   44|
+----------+-----+
only showing top 20 rows



1423

In [10]:
#2. Data Transformation

In [11]:
df_pivot=df.groupBy('BroadcastLogID').pivot ('CategoryID', ['24.0', '13', '26', '1']).count().show(10)
from pyspark.sql.functions import expr

df_unpivot=df.select('BroadcastLogID', expr("stack (2,'HELLO',ProgramClassID,'WORLD',CategoryID) as (PROGRAMCLASSID,count)")).show(10)

+--------------+----+----+----+----+
|BroadcastLogID|24.0|  13|  26|   1|
+--------------+----+----+----+----+
|    1237618818|NULL|NULL|   1|NULL|
|    1218443473|   1|NULL|NULL|NULL|
|    1213684274|NULL|NULL|NULL|NULL|
|    1214429129|   1|NULL|NULL|NULL|
|    1246303285|   1|NULL|NULL|NULL|
|    1214429280|   1|NULL|NULL|NULL|
|    1196158607|   1|NULL|NULL|NULL|
|    1213684669|   1|NULL|NULL|NULL|
|    1225956992|   1|NULL|NULL|NULL|
|    1205892355|   1|NULL|NULL|NULL|
+--------------+----+----+----+----+
only showing top 10 rows

+--------------+--------------+-----+
|BroadcastLogID|PROGRAMCLASSID|count|
+--------------+--------------+-----+
|    1196192316|         HELLO|   19|
|    1196192316|         WORLD|   13|
|    1196192317|         HELLO|   20|
|    1196192317|         WORLD|   24|
|    1196192318|         HELLO|    3|
|    1196192318|         WORLD|   24|
|    1196192319|         HELLO|    3|
|    1196192319|         WORLD|   24|
|    1196192320|         HELLO|    3|


In [12]:
# Window functions: Use window functions to calculate running totals, moving averages, and ranking.
from pyspark.sql import Window
from pyspark.sql import functions as fn
window_csvl=Window.orderBy('BroadcastLogID')

window_csv=Window.partitionBy('BroadcastLogID').orderBy ("LogServiceID")


print("Running Total")
df_csv_W= df.withColumn("running_total",fn.sum("SequenceNO").over(window_csvl))

df_csv_W.show(10)

print("Moving Average") 

windows_moving=Window.rowsBetween(-1,0)

df_csv_w_m=df.withColumn('Moving avg',fn.avg ('AudienceTargetAgeID').over(windows_moving)).show(10)

print("Ranking")

df_csv_rank=df.withColumn("rank",fn.rank().over (window_csv))


df_csv_rank.show(10)

Running Total
+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------+-------------------+--------------------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+-------------+
|BroadcastLogID|LogServiceID|   LogDate|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|           Duration|            EndTime|LogEntryDate|ProductionNO|        ProgramTitle|          StartTime|            Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|r

In [13]:
#3. Data Aggregation and Analysis 105 from pyspark.sql.types import


In [14]:
# Advanced aggregations: Group by multiple columns and perform aggregations like sum, average, min, max, and standard deviation.


agg_df= df.groupBy('BroadcastLogID', 'LogServiceID').agg (sum('AudienceTargetAgeID').alias('SUM_TOTAL'),\
                                                            min('CategoryID').alias('MIN_VALUE'),\
                                                            max('EthnicProgramID').alias('MAX VALUE'),\
                                                            avg ('AudienceTargetEthnicID').alias('AVERAGE TOTAL'),\
                                                            stddev('CountryOfOriginID').alias('STD_DEV'))


agg_df.show()




+--------------+------------+---------+---------+---------+-------------+-------+
|BroadcastLogID|LogServiceID|SUM_TOTAL|MIN_VALUE|MAX VALUE|AVERAGE TOTAL|STD_DEV|
+--------------+------------+---------+---------+---------+-------------+-------+
|    1196192362|        3157|        3|       24|        6|        120.0|   NULL|
|    1196193247|        3157|        3|       24|        6|        120.0|   NULL|
|    1196193344|        3157|        3|       24|        6|        120.0|   NULL|
|    1196705530|        3190|        3|       24|        6|        120.0|   NULL|
|    1237733886|        3191|        3|       27|        6|        120.0|   NULL|
|    1237734044|        3191|        3|       24|        6|        120.0|   NULL|
|    1237618903|        3192|        3|       26|        6|        120.0|   NULL|
|    1237619170|        3192|        3|       24|        6|        120.0|   NULL|
|    1237619442|        3192|        3|       26|        6|        120.0|   NULL|
|    1237619643|

In [15]:
# Join operations: Join the DataFrame, with another DataFrame on multiple keys with different. join types (inner, left, right, full outer)-

# inner join


print("INNER JOIN")
inner_df=df.join(df_csv_W,on=['BroadcastLogID', 'LogServiceID'],how='inner')

inner_df.show()

# left join
print("LEFT_JOIN")

left_df=df.join(df_csv_W,on=['BroadcastLogID', 'LogServiceID'],how='left')

left_df.show()

# right join
print('RIGHT JOIN')

right_df=df.join(df_csv_W,on=['BroadcastLogID', 'LogServiceID'],how='right')

right_df.show()

# outer join
print('OUTER_JOIN')

outer_df=df.join(df_csv_W,on=['BroadcastLogID', 'LogServiceID'],how='outer')

outer_df.show()




INNER JOIN
+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------+-------------------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------+-------------------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+-------------+
|BroadcastLogID|LogServiceID|   LogDate|SequenceNO|Audien

In [16]:
# 4. Data Cleaning and Validation



In [17]:
# String operations: Perform complex string manipulations and pattern matching using regular arr

from pyspark.sql.functions import *

df_clean=df.withColumn('ProgramTitle', regexp_replace('ProgramTitle', '3rd', "third"))

df_clean=df.withColumn('ProductionNO', regexp_replace('ProductionNO', 'A',""))

df_clean.show(truncate=False)

# Validate the data for various constraints (e.g., unique values, value ranges) and create a report of invalid records.

records= df_clean.groupBy('DubDramaCreditID').count().filter("count > 1")

min_val=4
max_val=10

range_df= records.filter((records['DubDramaCreditID'] < min_val) | (records['DubDramaCreditID'] > max_val))

range_df.show()



+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+---------------------------------------------+-------------------+--------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate   |SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration           |EndTime            |LogEntryDate|ProductionNO|ProgramTitle                                 |StartTime          |Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|
+-

In [18]:
# 5. Performance Optimization:

In [19]:
# Partitioning: Repartition the DataFrame based on a specific column to optimize the performance of subsequent operations.

print (f"Intial number of partitions: {df.rdd.getNumPartitions()}")

df_repartition=df.repartition (20,col ("BroadcastLogID"))

print (f"number of partitions after repartitioning {df_repartition.rdd.getNumPartitions()}")

df_repartition.show()

result=df_repartition.groupBy("LogServiceID").agg ({"AudienceTargetAgeID":"sum"})

result.show()


Intial number of partitions: 8
number of partitions after repartitioning 20
+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+--------------------+-------------------+--------------------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|   LogDate|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|           Duration|            EndTime|LogEntryDate|ProductionNO|        ProgramTitle|          StartTime|            Subtitle|NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|Composi

In [20]:
# 6. Data Visualization:
# Create visualizations: Use PySpark with a visualization library (e.g., Matplotlib, Seaborn) to create various plots like histograms, bar charts, and scatter plots.

In [21]:
df.createOrReplaceTempView("visuals")

In [22]:
sqldf = sc.sql("select * from visuals where BroadcastLogID>1198495531 and BroadcastLogID<1290000000 order by BroadcastLogID asc limit 15")
sqldf.show(truncate=False)

+--------------+------------+----------+----------+-------------------+----------------------+----------+---------------+-----------------+----------------+---------------+------------------+--------------+--------------------+------------+-------------------+-------------------+------------+------------+-------------------------+-------------------+------------+--------------------+------------------+----------------------+-------------+---------+---------+---------+---------+
|BroadcastLogID|LogServiceID|LogDate   |SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|CategoryID|ClosedCaptionID|CountryOfOriginID|DubDramaCreditID|EthnicProgramID|ProductionSourceID|ProgramClassID|FilmClassificationID|ExhibitionID|Duration           |EndTime            |LogEntryDate|ProductionNO|ProgramTitle             |StartTime          |Subtitle    |NetworkAffiliationID|SpecialAttentionID|BroadcastOriginPointID|CompositionID|Producer1|Producer2|Language1|Language2|
+--------------+------------+-----

In [23]:

import plotly.express as px
fig=px.bar(sqldf,y='CategoryID',x='ProgramTitle',title='Vis1', width=1000, height=400)
fig.show()

In [24]:
import plotly.express as px
fig=px.line(sqldf,y='CategoryID',x='ProgramTitle',title='Vis1', width=1000, height=400)
fig.show()

In [25]:
sqldf = sc.sql("select count(Producer1), Producer1 from visuals group by Producer1 order by count(Producer1) desc")
sqldf.show(50,truncate=False)

+----------------+---------+
|count(Producer1)|Producer1|
+----------------+---------+
|90823           |CTV      |
|19571           |ECG      |
|11468           |GLOBAL   |
|11399           |CBC      |
|8970            |CIII     |
|7709            |RDSINF   |
|7568            |CFTM     |
|7261            |SRC      |
|6079            |APTN     |
|5510            |RDI      |
|4823            |TWN      |
|3177            |NEWSW    |
|2595            |CBLFT    |
|2480            |CTV2     |
|2255            |CHAN     |
|2216            |OMON     |
|2208            |TFO      |
|2136            |CFCN     |
|2083            |CBXFT    |
|2061            |VISION   |
|1995            |CFRN     |
|1893            |IDNR     |
|1847            |CKCK     |
|1745            |CKTV     |
|1681            |CBOFT    |
|1590            |RDS      |
|1585            |CHEX     |
|1528            |CFQC     |
|1483            |CKTM     |
|1471            |CBVT     |
|1378            |CJOH     |
|1290         

In [26]:
fig = px.pie(sqldf, values='count(Producer1)', names='Producer1')
fig.update_traces(textposition='inside')
fig.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
fig.show()

In [27]:
sqldf = sc.sql("select count(ProgramTitle), ProgramTitle from visuals group by ProgramTitle order by count(ProgramTitle) desc limit 15")
sqldf.show(50,truncate=False)

+-------------------+------------------------------------------+
|count(ProgramTitle)|ProgramTitle                              |
+-------------------+------------------------------------------+
|4453               |15-SPECIALTY CHANNELS-Promotions Specialty|
|2410               |15-SPECIALTY CHANNELS-Canadian Generic    |
|1931               |15-GLOBAL TV TORONTO-Promotions Global    |
|1004               |Station Identification                    |
|897                |PROCTER & GAMBLE INC.-                    |
|781                |TD/SOCCERQUEENG                           |
|771                |3M CANADA/Household items                 |
|762                |Trivago                                   |
|740                |INDEED IRELAND/INTERNET/WEB SERV          |
|706                |Mars Canada Inc                           |
|636                |MONDELEZ/CADBURY DAIRY MILK/CHOC          |
|635                |Procter & Gamble Inc.                     |
|634                |CHAP

In [28]:
fig = px.pie(sqldf, values='count(ProgramTitle)', names='ProgramTitle')
fig.update_traces(textposition='inside')
fig.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
fig.show()